In [9]:
import numpy as np
import pandas as pd



In [10]:
df = pd.read_csv('market_sales_prediction.csv')
df.head(5)

,Order_ID,Date,Region,City,Product_Category,Product_Name,Price,Discount (%),Quantity,Customer_Age,Customer_Gender,Sales_Channel,Payment_Method,Season,Marketing_Spend,Competitor_Price,Stock_Availability,Delivery_Time,Sales
0,101800,03/06/2021,EAST,Patna,Electronics,Lapttop Pro,217.92,29.1,5.0,20,MALE,Online,Credit Card,Monsoon,154.23,226.44,Out of Stock,1.0,0.86
1,106625,2021-05-22,West,Surat,Furniture,Wooden Chair,1480.85,31.5,1.0,31,female,Offline,upi,Summer,17.88,1302.91,In stock,4.0,938.12
2,113555,2022-03-16,West,Ahmedabad,Furniture,Sofa Set,365.34,21.2,10.0,48,Male,Offline,Debit Card,Summer,68.29,340.68,In Stock,7.0,2236.50
3,102309,2021-03-29,West,Pune,Electronics,iPhone 14,1800.11,NaN,7.0,41,Female,Offline,upi,Summer,50.12,1551.02,out of stock,8.0,0.67
4,107837,29/11/2020,South,Hyderabad,Grocery,Wheat Flour,40.90,25.1,1.0,37,female,Offline,card,Festive,133.78,46.08,out of stock,6.0,0.63


In [11]:
df['Item_Identifier'].value_counts()

KeyError: 'Item_Identifier'

In [ ]:
df['Item_Type'] = df['Item_Identifier'].str[:2]
df['Item_Type'].value_counts()

Item_Type
FD    6125
NC    1599
DR     799
Name: count, dtype: int64

In [ ]:
df['Item_Type'] = df['Item_Type'].map({'FD' : 'Food', 'DR' : 'Drinks', 'NC' : 'Non-Consumable'})
df['Item_Type'].value_counts()

Item_Type
Food              6125
Non-Consumable    1599
Drinks             799
Name: count, dtype: int64

In [ ]:
df['Item_Type'].count()

np.int64(8523)

In [ ]:
df['Item_Weight'].isnull().sum()    

np.int64(1463)

In [ ]:
df['Item_Weight'].value_counts() 


Item_Weight
12.150    86
17.600    82
13.650    77
11.800    76
9.300     68
          ..
5.210      2
7.685      1
9.420      1
6.520      1
5.400      1
Name: count, Length: 415, dtype: int64

ValueError: could not convert string to float: 'FDA15'

In [12]:
# =============================================================================
# MARKET SALES PREDICTION — END-TO-END MACHINE LEARNING PIPELINE
# Senior ML Engineer | Production-Level Code
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    BaggingRegressor, StackingRegressor
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── Styling ──────────────────────────────────────────────────────────────────
PALETTE   = ["#0D1117", "#161B22", "#21262D", "#30363D",
             "#58A6FF", "#3FB950", "#F78166", "#FFA657", "#D2A8FF"]
ACCENT    = "#58A6FF"
GREEN     = "#3FB950"
RED       = "#F78166"
ORANGE    = "#FFA657"
PURPLE    = "#D2A8FF"
BG        = "#0D1117"
CARD      = "#161B22"
BORDER    = "#30363D"
TEXT      = "#E6EDF3"
SUBTEXT   = "#8B949E"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    CARD,
    "axes.edgecolor":    BORDER,
    "axes.labelcolor":   TEXT,
    "xtick.color":       SUBTEXT,
    "ytick.color":       SUBTEXT,
    "text.color":        TEXT,
    "grid.color":        BORDER,
    "grid.linewidth":    0.6,
    "font.family":       "monospace",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# STEP 1 — DATA LOADING & EXPLORATORY DATA ANALYSIS
# =============================================================================

def load_and_explore(filepath: str) -> pd.DataFrame:
    """Load CSV and print foundational EDA metrics."""
    print("\n" + "═" * 70)
    print("  STEP 1 │ DATA LOADING & EXPLORATORY DATA ANALYSIS")
    print("═" * 70)

    df = pd.read_csv(filepath, low_memory=False)

    print(f"\n📐 Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"💾 Memory usage   : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

    print("\n── dtypes & non-null counts ──────────────────────────────────────")
    print(df.info())

    print("\n── Statistical summary ───────────────────────────────────────────")
    print(df.describe(include='all').T.to_string())

    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print("\n── Missing values ────────────────────────────────────────────────")
    if missing.empty:
        print("   ✓ No missing values found.")
    else:
        pct = (missing / len(df) * 100).round(2)
        mv_df = pd.DataFrame({'Missing': missing, 'Pct (%)': pct})
        print(mv_df.to_string())

    # Duplicates
    dup_count = df.duplicated().sum()
    print(f"\n── Duplicate rows : {dup_count:,} ({dup_count/len(df)*100:.2f}%)")

    print("\n── Key Insights ─────────────────────────────────────────────────")
    numeric_df = df.select_dtypes(include=[np.number])
    if 'Sales' in numeric_df.columns:
        print(f"   • Target (Sales) range : {df['Sales'].min():.2f} – {df['Sales'].max():.2f}")
        print(f"   • Target (Sales) mean  : {df['Sales'].mean():.2f}  |  median : {df['Sales'].median():.2f}")
        print(f"   • Target (Sales) std   : {df['Sales'].std():.2f}")
        skew = df['Sales'].skew()
        print(f"   • Target (Sales) skew  : {skew:.3f}  {'(right-skewed)' if skew > 0 else '(left-skewed)'}")
    if 'Product_Category' in df.columns:
        print(f"\n   • Top category : {df['Product_Category'].mode()[0]}")
    if 'Season' in df.columns:
        print(f"   • Top season   : {df['Season'].mode()[0]}")

    return df


# =============================================================================
# STEP 1b — EDA VISUALISATIONS
# =============================================================================

def plot_eda(df: pd.DataFrame):
    """Generate and save comprehensive EDA visuals."""
    print("\n── Generating EDA visualizations ─────────────────────────────────")

    # --- Fig 1: Overview dashboard (distributions + boxplots) ---------------
    num_cols = ['Price', 'Discount (%)', 'Quantity', 'Marketing_Spend',
                'Competitor_Price', 'Delivery_Time', 'Sales']
    num_cols = [c for c in num_cols if c in df.columns]

    fig = plt.figure(figsize=(22, 16), facecolor=BG)
    fig.suptitle("Market Sales — Distribution & Outlier Overview",
                 color=TEXT, fontsize=16, fontweight='bold', y=0.98)

    n = len(num_cols)
    rows = 2
    cols = (n + 1) // 2

    for idx, col in enumerate(num_cols):
        ax = fig.add_subplot(rows, cols, idx + 1)
        data = df[col].dropna()
        ax.hist(data, bins=40, color=ACCENT, alpha=0.75, edgecolor='none')
        ax.axvline(data.mean(),   color=ORANGE, lw=1.5, ls='--', label='mean')
        ax.axvline(data.median(), color=GREEN,  lw=1.5, ls=':',  label='median')
        ax.set_title(col, color=TEXT, fontsize=10, fontweight='bold')
        ax.set_xlabel('')
        ax.legend(fontsize=7, labelcolor=SUBTEXT,
                  facecolor=CARD, edgecolor=BORDER)
        ax.tick_params(labelsize=7)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(f"{OUTPUT_DIR}/eda_distributions.png", dpi=150, bbox_inches='tight',
                facecolor=BG)
    plt.close()

    # --- Fig 2: Boxplots by category ----------------------------------------
    if 'Product_Category' in df.columns:
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=BG)
        fig.suptitle("Sales / Price / Discount by Product Category",
                     color=TEXT, fontsize=14, fontweight='bold')
        cats = df['Product_Category'].dropna().unique()
        box_colors = [ACCENT, GREEN, ORANGE, RED, PURPLE]
        for ax, metric in zip(axes, ['Sales', 'Price', 'Discount (%)']):
            if metric not in df.columns:
                continue
            data_by_cat = [df.loc[df['Product_Category'] == c, metric].dropna().values
                           for c in cats]
            bp = ax.boxplot(data_by_cat,
                            patch_artist=True,
                            medianprops=dict(color='white', linewidth=2),
                            whiskerprops=dict(color=SUBTEXT),
                            capprops=dict(color=SUBTEXT),
                            flierprops=dict(marker='o', markerfacecolor=RED,
                                           markersize=3, alpha=0.4))
            for patch, color in zip(bp['boxes'], box_colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            ax.set_xticklabels(cats, rotation=30, ha='right', fontsize=8)
            ax.set_title(metric, color=TEXT, fontsize=11, fontweight='bold')
        plt.tight_layout()
        fig.savefig(f"{OUTPUT_DIR}/eda_boxplots.png", dpi=150, bbox_inches='tight',
                    facecolor=BG)
        plt.close()

    # --- Fig 3: Correlation heatmap -----------------------------------------
    numeric_df = df.select_dtypes(include=[np.number]).dropna(axis=1, how='all')
    corr = numeric_df.corr()

    fig, ax = plt.subplots(figsize=(14, 11), facecolor=BG)
    cmap = sns.diverging_palette(220, 10, as_cmap=True)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
                linewidths=0.5, linecolor=BORDER,
                annot_kws={"size": 8, "color": TEXT},
                ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title("Feature Correlation Heatmap",
                 color=TEXT, fontsize=14, fontweight='bold', pad=15)
    ax.tick_params(labelsize=8, rotation=30)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/eda_correlation_heatmap.png", dpi=150,
                bbox_inches='tight', facecolor=BG)
    plt.close()

    # --- Fig 4: Bivariate — Sales vs key features ---------------------------
    fig, axes = plt.subplots(2, 3, figsize=(20, 11), facecolor=BG)
    fig.suptitle("Bivariate Analysis — Sales vs Key Features",
                 color=TEXT, fontsize=14, fontweight='bold')
    biv_features = ['Price', 'Discount (%)', 'Marketing_Spend',
                    'Quantity', 'Delivery_Time', 'Competitor_Price']
    biv_features = [f for f in biv_features if f in df.columns]
    axes_flat = axes.flatten()
    for i, feat in enumerate(biv_features):
        ax = axes_flat[i]
        sample = df[[feat, 'Sales']].dropna().sample(min(2000, len(df)),
                                                     random_state=42)
        ax.scatter(sample[feat], sample['Sales'],
                   alpha=0.35, s=10, color=ACCENT)
        # Trend line
        z = np.polyfit(sample[feat], sample['Sales'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(sample[feat].min(), sample[feat].max(), 100)
        ax.plot(x_line, p(x_line), color=ORANGE, lw=2)
        ax.set_xlabel(feat, fontsize=9)
        ax.set_ylabel('Sales', fontsize=9)
        ax.set_title(f"{feat} vs Sales", color=TEXT, fontsize=10, fontweight='bold')
    for j in range(len(biv_features), len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f"{OUTPUT_DIR}/eda_bivariate.png", dpi=150, bbox_inches='tight',
                facecolor=BG)
    plt.close()

    print("   ✓ EDA plots saved (4 figures).")


# =============================================================================
# STEP 2 — DATA PREPROCESSING
# =============================================================================

def standardize_text(val):
    """Strip, lowercase and normalize common categorical noise."""
    if pd.isna(val):
        return val
    v = str(val).strip().lower()
    # Gender
    if v in ('m',):     return 'male'
    if v in ('f',):     return 'female'
    # Stock
    if v in ('out', 'out of stock', 'outofstock'): return 'out of stock'
    if v in ('in stock', 'instock'):               return 'in stock'
    # Payment
    if 'credit' in v or v == 'card':  return 'credit card'
    if 'debit'  in v:                 return 'debit card'
    if 'upi'    in v:                 return 'upi'
    if 'cash'   in v:                 return 'cash'
    # Region
    if v in ('north','south','east','west'): return v
    return v


def parse_date_safe(val):
    """Parse dates that may be in YYYY-MM-DD or DD/MM/YYYY format."""
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.to_datetime(val, errors='coerce')


def remove_outliers_iqr(df: pd.DataFrame, cols: list,
                         factor: float = 3.0) -> pd.DataFrame:
    """Cap extreme outliers to [Q1 - factor*IQR, Q3 + factor*IQR]."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lo, hi = Q1 - factor * IQR, Q3 + factor * IQR
        df[col] = df[col].clip(lower=lo, upper=hi)
    return df


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Full preprocessing pipeline — returns cleaned DataFrame."""
    print("\n" + "═" * 70)
    print("  STEP 2 │ DATA PREPROCESSING")
    print("═" * 70)

    df = df.copy()

    # ── 2.1 Remove exact duplicates ─────────────────────────────────────────
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"\n[2.1] Duplicates removed   : {before - len(df):,}")

    # ── 2.2 Parse & expand Date column ──────────────────────────────────────
    df['Date_parsed'] = df['Date'].apply(parse_date_safe)
    df['Year']  = df['Date_parsed'].dt.year.astype('Int64')
    df['Month'] = df['Date_parsed'].dt.month.astype('Int64')
    df['Day']   = df['Date_parsed'].dt.day.astype('Int64')
    df.drop(columns=['Date', 'Date_parsed'], inplace=True)
    print("[2.2] Date parsed → Year / Month / Day columns added.")

    # ── 2.3 Standardize text / categoricals ─────────────────────────────────
    text_cols = ['Region', 'City', 'Product_Category', 'Product_Name',
                 'Customer_Gender', 'Sales_Channel', 'Payment_Method',
                 'Season', 'Stock_Availability']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(standardize_text)
    print("[2.3] Text columns standardized.")

    # ── 2.4 Fix known bad values ─────────────────────────────────────────────
    # Discount: clamp to [0, 100]
    if 'Discount (%)' in df.columns:
        df['Discount (%)'] = pd.to_numeric(df['Discount (%)'], errors='coerce')
        df.loc[df['Discount (%)'] < 0,   'Discount (%)'] = np.nan
        df.loc[df['Discount (%)'] > 100, 'Discount (%)'] = np.nan

    # Customer Age: realistic range
    if 'Customer_Age' in df.columns:
        df['Customer_Age'] = pd.to_numeric(df['Customer_Age'], errors='coerce')
        df.loc[(df['Customer_Age'] < 10) | (df['Customer_Age'] > 100),
               'Customer_Age'] = np.nan

    # Delivery Time: realistic range
    if 'Delivery_Time' in df.columns:
        df['Delivery_Time'] = pd.to_numeric(df['Delivery_Time'], errors='coerce')
        df.loc[(df['Delivery_Time'] < 1) | (df['Delivery_Time'] > 20),
               'Delivery_Time'] = np.nan
    print("[2.4] Bad / unrealistic values nullified.")

    # ── 2.5 Missing value imputation ─────────────────────────────────────────
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols  = df.select_dtypes(include=['object']).columns.tolist()

    for col in num_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

    for col in cat_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()
            if not mode_val.empty:
                df[col].fillna(mode_val[0], inplace=True)

    print(f"[2.5] Missing values imputed (numeric→median, categorical→mode).")
    print(f"      Remaining nulls : {df.isnull().sum().sum()}")

    # ── 2.6 Outlier treatment (IQR capping) ──────────────────────────────────
    outlier_cols = ['Price', 'Marketing_Spend', 'Competitor_Price',
                    'Sales', 'Discount (%)']
    df = remove_outliers_iqr(df, outlier_cols, factor=3.0)
    print("[2.6] Outliers capped via IQR (factor=3).")

    print(f"\n   ✓ Preprocessing complete — shape: {df.shape}")
    return df


# =============================================================================
# STEP 3 — FEATURE ENGINEERING
# =============================================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create domain-driven, model-ready features."""
    print("\n" + "═" * 70)
    print("  STEP 3 │ FEATURE ENGINEERING")
    print("═" * 70)

    df = df.copy()

    # Price after discount
    if 'Price' in df.columns and 'Discount (%)' in df.columns:
        df['Price_after_discount'] = (
            df['Price'] * (1 - df['Discount (%)'] / 100)
        ).round(2)

    # Revenue proxy
    if 'Price' in df.columns and 'Quantity' in df.columns:
        df['Revenue'] = (df['Price'] * df['Quantity']).round(2)

    # Discount impact (absolute saving per unit)
    if 'Price' in df.columns and 'Discount (%)' in df.columns:
        df['Discount_impact'] = (
            df['Price'] * df['Discount (%)'] / 100
        ).round(2)

    # Delivery speed category  (1-3 fast, 4-6 medium, 7+ slow)
    if 'Delivery_Time' in df.columns:
        df['Delivery_speed_category'] = pd.cut(
            df['Delivery_Time'],
            bins=[0, 3, 6, 100],
            labels=['fast', 'medium', 'slow']
        ).astype(str)

    # Seasonal flag  (Festive = 1, else 0)
    if 'Season' in df.columns:
        df['Seasonal_flag'] = (df['Season'] == 'festive').astype(int)

    # Customer age group
    if 'Customer_Age' in df.columns:
        df['Customer_age_group'] = pd.cut(
            df['Customer_Age'],
            bins=[0, 25, 35, 50, 200],
            labels=['young', 'adult', 'middle_aged', 'senior']
        ).astype(str)

    # Price-to-competitor ratio
    if 'Price' in df.columns and 'Competitor_Price' in df.columns:
        df['Price_vs_competitor'] = (
            df['Price'] / (df['Competitor_Price'] + 1e-6)
        ).round(4)

    # Marketing efficiency
    if 'Marketing_Spend' in df.columns and 'Revenue' in df.columns:
        df['Marketing_efficiency'] = (
            df['Revenue'] / (df['Marketing_Spend'] + 1e-6)
        ).round(4)

    new_feats = [
        'Price_after_discount', 'Revenue', 'Discount_impact',
        'Delivery_speed_category', 'Seasonal_flag', 'Customer_age_group',
        'Price_vs_competitor', 'Marketing_efficiency'
    ]
    created = [f for f in new_feats if f in df.columns]
    print(f"   ✓ {len(created)} new features created: {created}")
    return df


# =============================================================================
# STEP 2b — ENCODING (after feature engineering so new cat cols are included)
# =============================================================================

def encode_and_scale(df: pd.DataFrame, target: str = 'Sales'):
    """
    Encode categoricals (Label Encoding for low-cardinality,
    One-Hot for multi-class), then StandardScale numerics.

    Returns X_scaled (np.ndarray), y (np.ndarray), feature_names, scaler.
    """
    print("\n── Encoding & Scaling ────────────────────────────────────────────")

    df = df.copy()

    # Drop Order_ID (identifier — no predictive signal)
    drop_cols = ['Order_ID']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # Separate target
    y = df[target].values
    X = df.drop(columns=[target])

    # Label-encode binary / ordinal categoricals
    label_enc_cols = ['Sales_Channel', 'Stock_Availability',
                      'Seasonal_flag', 'Delivery_speed_category',
                      'Customer_age_group']
    for col in label_enc_cols:
        if col in X.columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))

    # One-Hot encode remaining object columns
    ohe_cols = X.select_dtypes(include=['object']).columns.tolist()
    if ohe_cols:
        X = pd.get_dummies(X, columns=ohe_cols, drop_first=True)
        print(f"   One-Hot encoded : {ohe_cols}")

    # Ensure all numeric
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

    feature_names = X.columns.tolist()

    # Standard scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print(f"   ✓ Final feature matrix : {X_scaled.shape}")
    return X_scaled, y, feature_names, scaler, X.columns.tolist()


# =============================================================================
# STEP 4 — TRAIN/TEST SPLIT
# =============================================================================

def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "═" * 70)
    print("  STEP 4 │ TRAIN-TEST SPLIT")
    print("═" * 70)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    print(f"   Train : {X_train.shape[0]:,} samples")
    print(f"   Test  : {X_test.shape[0]:,} samples")
    return X_train, X_test, y_train, y_test


# =============================================================================
# STEP 5 — TRAIN & EVALUATE MULTIPLE MODELS
# =============================================================================

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Fit model, predict, and return metrics dict."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return {
        'Model': name,
        'R²': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        '_model': model,
        '_preds': y_pred
    }


def train_all_models(X_train, X_test, y_train, y_test):
    """Train Linear Regression, Decision Tree, RF, Gradient Boosting."""
    print("\n" + "═" * 70)
    print("  STEP 5 │ TRAINING MULTIPLE ML MODELS")
    print("═" * 70)

    models = {
        'Linear Regression':    LinearRegression(),
        'Decision Tree':        DecisionTreeRegressor(
                                    max_depth=10, min_samples_leaf=20,
                                    random_state=42),
        'Random Forest':        RandomForestRegressor(
                                    n_estimators=150, max_depth=15,
                                    min_samples_leaf=10, n_jobs=-1,
                                    random_state=42),
        'Gradient Boosting':    GradientBoostingRegressor(
                                    n_estimators=150, learning_rate=0.08,
                                    max_depth=5, random_state=42),
    }

    # Try XGBoost if available
    try:
        from xgboost import XGBRegressor
        models['XGBoost'] = XGBRegressor(
            n_estimators=150, learning_rate=0.08, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            verbosity=0, random_state=42)
        print("   XGBoost detected — added to model pool.")
    except ImportError:
        print("   XGBoost not installed — skipping.")

    results = []
    for name, model in models.items():
        res = evaluate_model(name, model, X_train, X_test, y_train, y_test)
        print(f"   [{name:<22}]  R²={res['R²']:.4f}  "
              f"MAE={res['MAE']:>9.2f}  RMSE={res['RMSE']:>9.2f}")
        results.append(res)

    return results, models


# =============================================================================
# STEP 6 — ENSEMBLE LEARNING
# =============================================================================

def ensemble_learning(X_train, X_test, y_train, y_test,
                       base_models: dict, results: list):
    """Bagging, Boosting summary, and Stacking Regressor."""
    print("\n" + "═" * 70)
    print("  STEP 6 │ ENSEMBLE LEARNING")
    print("═" * 70)

    ensemble_results = []

    # ── Bagging (extra BaggingRegressor wrapping DT) ─────────────────────────
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=8, random_state=42),
        n_estimators=100, max_samples=0.8, max_features=0.8,
        random_state=42, n_jobs=-1
    )
    res = evaluate_model('Bagging (DT base)', bagging,
                         X_train, X_test, y_train, y_test)
    print(f"   [{'Bagging (DT base)':<22}]  R²={res['R²']:.4f}  "
          f"MAE={res['MAE']:>9.2f}  RMSE={res['RMSE']:>9.2f}")
    ensemble_results.append(res)

    # ── Stacking Regressor ────────────────────────────────────────────────────
    estimators = [
        ('dt',  DecisionTreeRegressor(max_depth=8, random_state=42)),
        ('rf',  RandomForestRegressor(n_estimators=80, max_depth=10,
                                       n_jobs=-1, random_state=42)),
        ('gb',  GradientBoostingRegressor(n_estimators=80,
                                          learning_rate=0.1, random_state=42)),
    ]
    stacking = StackingRegressor(
        estimators=estimators,
        final_estimator=LinearRegression(),
        cv=5, n_jobs=-1
    )
    res = evaluate_model('Stacking Regressor', stacking,
                         X_train, X_test, y_train, y_test)
    print(f"   [{'Stacking Regressor':<22}]  R²={res['R²']:.4f}  "
          f"MAE={res['MAE']:>9.2f}  RMSE={res['RMSE']:>9.2f}")
    ensemble_results.append(res)

    return ensemble_results


# =============================================================================
# STEP 7 — MODEL COMPARISON & VISUALISATION
# =============================================================================

def compare_models(all_results: list):
    """Print leaderboard table and save comparison chart."""
    print("\n" + "═" * 70)
    print("  STEP 7 │ MODEL COMPARISON LEADERBOARD")
    print("═" * 70)

    compare_df = pd.DataFrame([
        {k: v for k, v in r.items() if not k.startswith('_')}
        for r in all_results
    ]).sort_values('R²', ascending=False).reset_index(drop=True)

    compare_df.index += 1
    print("\n" + compare_df.to_string(index=True))

    best = compare_df.iloc[0]
    print(f"\n   🏆 Best model : {best['Model']}")
    print(f"      R²={best['R²']}  MAE={best['MAE']}  RMSE={best['RMSE']}")

    # ── Plot comparison ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 7), facecolor=BG)
    fig.suptitle("Model Comparison — R² / MAE / RMSE",
                 color=TEXT, fontsize=15, fontweight='bold', y=1.01)

    metrics    = ['R²', 'MAE', 'RMSE']
    bar_colors = [GREEN, ORANGE, RED]

    for ax, metric, color in zip(axes, metrics, bar_colors):
        asc   = metric != 'R²'
        sorted_df = compare_df.sort_values(metric, ascending=asc)
        bars = ax.barh(sorted_df['Model'], sorted_df[metric],
                       color=color, alpha=0.8, edgecolor='none', height=0.6)
        for bar, val in zip(bars, sorted_df[metric]):
            ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                    f'{val:.3f}', va='center', fontsize=8, color=TEXT)
        ax.set_title(metric, color=TEXT, fontsize=12, fontweight='bold')
        ax.set_xlabel(metric, fontsize=9)
        ax.tick_params(labelsize=8)
        ax.invert_yaxis()

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150,
                bbox_inches='tight', facecolor=BG)
    plt.close()

    # ── Feature importance (best tree-based model) ───────────────────────────
    best_model_obj = next(
        (r['_model'] for r in all_results if r['Model'] == best['Model']), None
    )
    if best_model_obj and hasattr(best_model_obj, 'feature_importances_'):
        return compare_df, best_model_obj
    # Fallback: first model with feature_importances_
    for r in all_results:
        if hasattr(r['_model'], 'feature_importances_'):
            return compare_df, r['_model']

    return compare_df, None


def plot_feature_importance(model, feature_names: list, top_n: int = 20):
    """Save top-N feature importance bar chart."""
    if model is None:
        return
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1][:top_n]
    top_feats  = [feature_names[i] for i in idx]
    top_imps   = importances[idx]

    fig, ax = plt.subplots(figsize=(12, 8), facecolor=BG)
    bars = ax.barh(top_feats[::-1], top_imps[::-1],
                   color=ACCENT, alpha=0.85, edgecolor='none')
    for bar, imp in zip(bars, top_imps[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{imp:.4f}', va='center', fontsize=8, color=TEXT)
    ax.set_title(f"Top {top_n} Feature Importances",
                 color=TEXT, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance', fontsize=10)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=150,
                bbox_inches='tight', facecolor=BG)
    plt.close()
    print("   ✓ Feature importance chart saved.")


def plot_predictions_vs_actual(all_results: list, y_test: np.ndarray,
                                best_name: str):
    """Scatter actual vs predicted for the best model."""
    best_res = next((r for r in all_results if r['Model'] == best_name), None)
    if best_res is None:
        return
    y_pred = best_res['_preds']

    fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
    fig.suptitle(f"Prediction Analysis — {best_name}",
                 color=TEXT, fontsize=14, fontweight='bold')

    # Actual vs Predicted scatter
    ax = axes[0]
    ax.scatter(y_test, y_pred, alpha=0.3, s=8, color=ACCENT)
    lims = [min(y_test.min(), y_pred.min()),
            max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, color=ORANGE, lw=2, ls='--', label='Perfect fit')
    ax.set_xlabel('Actual Sales',    fontsize=10)
    ax.set_ylabel('Predicted Sales', fontsize=10)
    ax.set_title('Actual vs Predicted', color=TEXT, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, labelcolor=TEXT, facecolor=CARD, edgecolor=BORDER)

    # Residuals distribution
    ax2 = axes[1]
    residuals = y_test - y_pred
    ax2.hist(residuals, bins=50, color=PURPLE, alpha=0.8, edgecolor='none')
    ax2.axvline(0, color=ORANGE, lw=2, ls='--')
    ax2.set_xlabel('Residual (Actual − Predicted)', fontsize=10)
    ax2.set_ylabel('Frequency', fontsize=10)
    ax2.set_title('Residual Distribution', color=TEXT, fontsize=11, fontweight='bold')

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/predictions_vs_actual.png", dpi=150,
                bbox_inches='tight', facecolor=BG)
    plt.close()
    print("   ✓ Prediction vs actual chart saved.")


# =============================================================================
# STEP 8 — FINAL PREDICTION SYSTEM (sample inference)
# =============================================================================

def predict_new_sample(df_clean: pd.DataFrame,
                        best_model,
                        feature_names: list,
                        scaler: StandardScaler):
    """
    Demonstrate end-to-end inference on a single new observation.
    We build a realistic sample, run the same preprocessing & feature
    engineering, then predict Sales.
    """
    print("\n" + "═" * 70)
    print("  STEP 8 │ FINAL PREDICTION SYSTEM — NEW SAMPLE INFERENCE")
    print("═" * 70)

    # --- Build a single new raw row ─────────────────────────────────────────
    new_raw = pd.DataFrame([{
        'Order_ID':           999999,
        'Date':               '2025-11-05',        # Festive season
        'Region':             'North',
        'City':               'Delhi',
        'Product_Category':   'Electronics',
        'Product_Name':       'Samsung TV',
        'Price':              850.00,
        'Discount (%)':       20.0,
        'Quantity':           2,
        'Customer_Age':       34,
        'Customer_Gender':    'Male',
        'Sales_Channel':      'Online',
        'Payment_Method':     'Credit Card',
        'Season':             'Festive',
        'Marketing_Spend':    200.0,
        'Competitor_Price':   900.0,
        'Stock_Availability': 'In Stock',
        'Delivery_Time':      3,
        'Sales':              0.0         # placeholder — will be predicted
    }])

    print("\n   Input sample:")
    print(new_raw.drop(columns=['Sales']).T.to_string())

    # ── Apply same preprocessing ──────────────────────────────────────────────
    new_raw['Date_parsed'] = new_raw['Date'].apply(parse_date_safe)
    new_raw['Year']  = new_raw['Date_parsed'].dt.year.astype('Int64')
    new_raw['Month'] = new_raw['Date_parsed'].dt.month.astype('Int64')
    new_raw['Day']   = new_raw['Date_parsed'].dt.day.astype('Int64')
    new_raw.drop(columns=['Date', 'Date_parsed'], inplace=True)

    text_cols = ['Region', 'City', 'Product_Category', 'Product_Name',
                 'Customer_Gender', 'Sales_Channel', 'Payment_Method',
                 'Season', 'Stock_Availability']
    for col in text_cols:
        if col in new_raw.columns:
            new_raw[col] = new_raw[col].apply(standardize_text)

    # ── Apply same feature engineering ───────────────────────────────────────
    new_raw['Price_after_discount'] = new_raw['Price'] * (1 - new_raw['Discount (%)'] / 100)
    new_raw['Revenue']              = new_raw['Price'] * new_raw['Quantity']
    new_raw['Discount_impact']      = new_raw['Price'] * new_raw['Discount (%)'] / 100
    new_raw['Delivery_speed_category'] = pd.cut(
        new_raw['Delivery_Time'], bins=[0, 3, 6, 100],
        labels=['fast', 'medium', 'slow']).astype(str)
    new_raw['Seasonal_flag']        = (new_raw['Season'] == 'festive').astype(int)
    new_raw['Customer_age_group']   = pd.cut(
        new_raw['Customer_Age'], bins=[0, 25, 35, 50, 200],
        labels=['young', 'adult', 'middle_aged', 'senior']).astype(str)
    new_raw['Price_vs_competitor']  = new_raw['Price'] / (new_raw['Competitor_Price'] + 1e-6)
    new_raw['Marketing_efficiency'] = new_raw['Revenue'] / (new_raw['Marketing_Spend'] + 1e-6)

    # ── Encode to match training feature space ────────────────────────────────
    # Label encode same columns
    label_enc_cols = ['Sales_Channel', 'Stock_Availability',
                      'Seasonal_flag', 'Delivery_speed_category',
                      'Customer_age_group']
    for col in label_enc_cols:
        if col in new_raw.columns:
            le = LabelEncoder()
            # Fit on training distribution to keep same mapping
            new_raw[col] = le.fit_transform(new_raw[col].astype(str))

    # One-Hot encode remaining objects
    ohe_cols = new_raw.select_dtypes(include=['object']).columns.tolist()
    if 'Order_ID' in ohe_cols: ohe_cols.remove('Order_ID')
    if 'Sales'    in ohe_cols: ohe_cols.remove('Sales')
    if ohe_cols:
        new_raw = pd.get_dummies(new_raw, columns=ohe_cols, drop_first=True)

    # Drop Order_ID & target
    new_raw.drop(columns=[c for c in ['Order_ID', 'Sales'] if c in new_raw.columns],
                 inplace=True)

    # Align columns to training set
    for col in feature_names:
        if col not in new_raw.columns:
            new_raw[col] = 0
    new_raw = new_raw[feature_names]
    new_raw = new_raw.apply(pd.to_numeric, errors='coerce').fillna(0)

    # Scale
    X_new = scaler.transform(new_raw)

    # ── Predict ────────────────────────────────────────────────────────────────
    predicted_sales = best_model.predict(X_new)[0]
    print(f"\n   ✅ Predicted Sales : ₹ {predicted_sales:,.2f}")
    return predicted_sales


# =============================================================================
# MAIN — ORCHESTRATION
# =============================================================================

def main():
    CSV_PATH = "D:\\Sem-6\\ML Project\\outputs\\market_sales_prediction.csv"
    TARGET   = 'Sales'

    # ── Step 1: Load & EDA ───────────────────────────────────────────────────
    df_raw = load_and_explore(CSV_PATH)
    plot_eda(df_raw)

    # ── Step 2: Preprocess ───────────────────────────────────────────────────
    df_clean = preprocess(df_raw)

    # ── Step 3: Feature Engineering ──────────────────────────────────────────
    df_feat = engineer_features(df_clean)

    # ── Encode & Scale ───────────────────────────────────────────────────────
    X, y, feature_names, scaler, col_names = encode_and_scale(df_feat, target=TARGET)

    # ── Step 4: Split ────────────────────────────────────────────────────────
    X_train, X_test, y_train, y_test = split_data(X, y)

    # ── Step 5: Train base models ─────────────────────────────────────────────
    base_results, base_models = train_all_models(X_train, X_test, y_train, y_test)

    # ── Step 6: Ensemble ─────────────────────────────────────────────────────
    ensemble_results = ensemble_learning(
        X_train, X_test, y_train, y_test, base_models, base_results
    )

    # ── Step 7: Compare ──────────────────────────────────────────────────────
    all_results  = base_results + ensemble_results
    compare_df, best_model_obj = compare_models(all_results)

    best_name = compare_df.iloc[0]['Model']
    plot_feature_importance(best_model_obj, feature_names)
    plot_predictions_vs_actual(all_results, y_test, best_name)

    # ── Step 8: Predict on new sample ─────────────────────────────────────────
    best_model_full = next(
        r['_model'] for r in all_results if r['Model'] == best_name
    )
    predict_new_sample(df_clean, best_model_full, col_names, scaler)

    print("\n" + "═" * 70)
    print("  PIPELINE COMPLETE ✓")
    print(f"  All artefacts saved to: {OUTPUT_DIR}")
    print("═" * 70 + "\n")


if __name__ == '__main__':
    main()


══════════════════════════════════════════════════════════════════════
  STEP 1 │ DATA LOADING & EXPLORATORY DATA ANALYSIS
══════════════════════════════════════════════════════════════════════


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Sem-6\\ML Project\\outputs\\market_sales_prediction.csv'